<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/CNS_Hardened_End_to_End_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Circuittelligence Nervous System (CNS) - Hardened Pipeline V1.0
PROPRIETARY AND CONFIDENTIAL

Unified Engine: Intake (Sensory Input) + Core (AEP Knowledge Decay)
"""

from pydantic import BaseModel, Field
from typing import Literal, List, Dict, Set, Optional, Tuple
from datetime import datetime, timezone
import hashlib
import uuid

# ==========================================
# 1. FORMAL CONSTANTS (MATH & THRESHOLDS)
# ==========================================
# Intake Constants
SEMANTIC_DEDUPE_THRESHOLD = 0.95
PIR_HOT_THRESHOLD = 0.4
TRUST_TIER_WEIGHTS = {5: 1.0, 4: 0.8, 3: 0.5, 2: 0.2, 1: 0.05}

# Core AEP Constants
DECAY_RATE = 0.05
REINFORCEMENT_GAIN = 0.08
CONTRADICTION_PENALTY = 0.10
JULES_MULTIPLIER = 1.5

# Promotion Board Constants
PROMOTION_AUTHORITY_THRESHOLD = 0.65
PROMOTION_DIVERSITY_THRESHOLD = 3
PROMOTION_PERSISTENCE_THRESHOLD = 3


# ==========================================
# 2. UNIFIED PYDANTIC SCHEMAS
# ==========================================
class SourceRegistry(BaseModel):
    source_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    name: str
    source_type: Literal["rss", "api", "scraper", "manual", "internal", "jules_agent"]
    trust_tier: int = Field(ge=1, le=5)

class PIRDefinition(BaseModel):
    pir_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    keywords: List[str]

class IntakeSignal(BaseModel):
    signal_id: str
    source_id: uuid.UUID
    url: str
    title: str
    raw_text: str

class EvidenceEvent(BaseModel):
    event_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    claim_id: uuid.UUID
    context_id: uuid.UUID
    source_type: str
    reinforcement: float = 0.0
    contradiction: float = 0.0
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class Claim(BaseModel):
    id: uuid.UUID = Field(default_factory=uuid.uuid4)
    content: str
    authority_score: float = Field(default=0.1, ge=0.0, le=1.0)
    decay_survivals: int = Field(default=0)
    distinct_sources: Set[uuid.UUID] = Field(default_factory=set)
    is_promoted: bool = Field(default=False)
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    updated_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))


# ==========================================
# 3. SUBSYSTEMS
# ==========================================
class MockDatabase:
    """Shared state for the hardened pipeline."""
    def __init__(self):
        self.hashes: Set[str] = set()
        self.embeddings: Dict[uuid.UUID, str] = {} # claim_id -> text
        self.claims: Dict[uuid.UUID, Claim] = {}

    def semantic_search(self, text: str) -> Tuple[Optional[uuid.UUID], float]:
        for claim_id, stored_text in self.embeddings.items():
            if text.lower() == stored_text.lower():
                return claim_id, 1.0
            if text[:25] == stored_text[:25]: # Mock partial match
                return claim_id, 0.96
        return None, 0.0

class CoreSubsystem:
    """AEP Knowledge Decay and Promotion Board"""
    def __init__(self, db: MockDatabase):
        self.db = db

    def apply_decay_cycle(self):
        print("\n[CORE] --- Executing System-Wide Decay Cycle ---")
        for claim in self.db.claims.values():
            if claim.is_promoted: continue

            old_score = claim.authority_score
            claim.authority_score = max(0.0, claim.authority_score * (1 - DECAY_RATE))
            claim.decay_survivals += 1
            print(f"  -> Decayed Claim [{claim.id}]: {old_score:.3f} -> {claim.authority_score:.3f}")

    def process_evidence(self, event: EvidenceEvent):
        claim = self.db.claims.get(event.claim_id)
        if not claim: return

        j_mult = JULES_MULTIPLIER if event.source_type == "jules_agent" else 1.0
        gain = event.reinforcement * REINFORCEMENT_GAIN
        penalty = event.contradiction * CONTRADICTION_PENALTY * j_mult

        old_score = claim.authority_score
        claim.authority_score = max(0.0, min(1.0, old_score + gain - penalty))
        claim.distinct_sources.add(event.context_id)
        claim.updated_at = datetime.now(timezone.utc)

        print(f"[CORE] Evidence Processed | Auth: {old_score:.3f} -> {claim.authority_score:.3f} | Sources: {len(claim.distinct_sources)}")

    def evaluate_promotion(self, claim_id: uuid.UUID):
        claim = self.db.claims.get(claim_id)
        if not claim or claim.is_promoted: return

        if (claim.authority_score >= PROMOTION_AUTHORITY_THRESHOLD and
            len(claim.distinct_sources) >= PROMOTION_DIVERSITY_THRESHOLD and
            claim.decay_survivals >= PROMOTION_PERSISTENCE_THRESHOLD):
            claim.is_promoted = True
            print(f"[CORE] ★★★ PROMOTION BOARD APPROVED ★★★ Claim [{claim.id}] is now a Focused Goal.")

class IntakeSubsystem:
    """Sensory Input, Deduplication, and PIR Routing"""
    def __init__(self, db: MockDatabase, core: CoreSubsystem, sources: Dict[uuid.UUID, SourceRegistry], pirs: List[PIRDefinition]):
        self.db = db
        self.core = core
        self.sources = sources
        self.pirs = pirs

    def ingest(self, signal: IntakeSignal):
        print(f"\n[INTAKE] Ingesting: '{signal.title}'")
        source = self.sources.get(signal.source_id)
        if not source: return

        # 1. Exact Deduplication
        content_hash = hashlib.sha256(f"{signal.url}|{signal.raw_text}".encode()).hexdigest()
        if content_hash in self.db.hashes:
            print("  -> L1 Dedupe: Exact match dropped.")
            return
        self.db.hashes.add(content_hash)

        # 2. PIR Alignment
        pir_score = max([sum(1 for kw in p.keywords if kw in signal.raw_text.lower()) * 0.3 for p in self.pirs] + [0])
        if pir_score <= PIR_HOT_THRESHOLD:
            print("  -> PIR Gate: Low alignment. Routing to Cold Storage.")
            return

        # 3. Semantic Deduplication & Routing
        matched_claim_id, sim_score = self.db.semantic_search(signal.raw_text)

        reinforcement_val = TRUST_TIER_WEIGHTS[source.trust_tier]

        if sim_score >= SEMANTIC_DEDUPE_THRESHOLD and matched_claim_id:
            print(f"  -> L2 Dedupe: Semantic Match. Emitting Reinforcement to Claim [{matched_claim_id}]")
            ev = EvidenceEvent(
                claim_id=matched_claim_id, context_id=source.source_id,
                source_type=source.source_type, reinforcement=reinforcement_val
            )
            self.core.process_evidence(ev)
            self.core.evaluate_promotion(matched_claim_id)
        else:
            print("  -> L2 Dedupe: Novel Signal. Creating new Claim.")
            new_claim = Claim(content=signal.raw_text)
            self.db.claims[new_claim.id] = new_claim
            self.db.embeddings[new_claim.id] = signal.raw_text

            ev = EvidenceEvent(
                claim_id=new_claim.id, context_id=source.source_id,
                source_type=source.source_type, reinforcement=reinforcement_val
            )
            self.core.process_evidence(ev)


# ==========================================
# 4. ORCHESTRATION DEMO
# ==========================================
if __name__ == "__main__":
    # Setup infrastructure
    db = MockDatabase()
    core = CoreSubsystem(db)

    src_tier5 = SourceRegistry(name="SEC EDGAR", source_type="api", trust_tier=5)
    src_tier4 = SourceRegistry(name="WSJ", source_type="rss", trust_tier=4)
    src_tier2 = SourceRegistry(name="Twitter", source_type="scraper", trust_tier=2)
    src_jules = SourceRegistry(name="Jules Agent", source_type="jules_agent", trust_tier=5)

    sources = {s.source_id: s for s in [src_tier5, src_tier4, src_tier2, src_jules]}
    pirs = [PIRDefinition(keywords=["quantum", "encryption", "threat"])]

    intake = IntakeSubsystem(db, core, sources, pirs)

    # Sequence 1: Ingest Novel Signal
    intake.ingest(IntakeSignal(signal_id="1", source_id=src_tier5.source_id, url="sec.gov/1", title="Q3", raw_text="Quantum encryption threat detected."))

    # Sequence 2: Ingest Semantic Duplicate (Different Source)
    intake.ingest(IntakeSignal(signal_id="2", source_id=src_tier4.source_id, url="wsj.com/q", title="Quantum", raw_text="Quantum encryption threat detected by researchers."))

    # Sequence 3: Apply Time
    core.apply_decay_cycle()
    core.apply_decay_cycle()
    core.apply_decay_cycle() # Now passes persistence gate

    # Sequence 4: Third Source confirms (Passes Diversity Gate & Authority Gate)
    intake.ingest(IntakeSignal(signal_id="3", source_id=src_tier2.source_id, url="x.com/1", title="Leak", raw_text="Quantum encryption threat is real guys."))

    # Sequence 5: Adversarial Intervention (Jules finds contradiction)
    claim_id = list(db.claims.keys())[0]
    print("\n[JULES] Executing Counter-Intelligence...")
    jules_ev = EvidenceEvent(claim_id=claim_id, context_id=src_jules.source_id, source_type="jules_agent", contradiction=1.0)
    core.process_evidence(jules_ev)